# RoBERTa Fine-Tuning — Semantically Deduplicated Dataset

This is the **clean, final run**. RoBERTa-base fine-tuned on the semantically deduplicated WELFake dataset.

## Motivation

After completing notebook 02, the dataset was investigated further. A large number of articles were found with cosine similarity > 0.98 (the same story reworded slightly, which exact string deduplication cannot detect). These near-duplicates created a subtle form of data leakage: the model sees nearly identical articles during training and encounters them again at test time.

**Deduplication pipeline:**
1. Raw WELFake: 72,134 articles
2. Exact `(combined, label)` dedup → 63,676 articles  
3. Sentence Transformer embedding + cosine similarity threshold (> 0.98) → **63,074 articles**

Both the TF-IDF baseline (notebook 01) and this run use the same 63,074-article dataset for a fair comparison.

**Label mapping:** `0` = Fake, `1` = Real  
**Platform:** Local (GPU)

## Install Dependencies

In [1]:
!pip install transformers datasets accelerate -q

## Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import DataCollatorWithPadding
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import torch

## Load Semantically Deduplicated Dataset

Loading `final_dataset.csv` — the output of the Sentence Transformer deduplication pipeline. This file is **not** raw WELFake; it has already had both exact and semantic duplicates removed.

In [3]:
df = pd.read_csv("C:/Users/pabit/Downloads/final_dataset.csv")
print("After exact dedup:", len(df))

After exact dedup: 63074


## Train / Validation / Test Split

70/15/15 stratified split with the same `random_state=42` as notebook 02, ensuring the comparison is controlled.

Split sizes: 44,151 train / 9,461 val / 9,462 test (slightly smaller than notebook 02's 44,573/9,551/9,552 due to the additional ~600 articles removed by semantic dedup).

In [4]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])

print(len(train_df), len(val_df), len(test_df))

44151 9461 9462


## Leakage Check

Zero overlap confirmed between train and val/test splits on the `combined` field.

In [5]:
train_set = set(train_df["combined"])

print("Train-Val overlap:",
      val_df["combined"].isin(train_set).sum())

print("Train-Test overlap:",
      test_df["combined"].isin(train_set).sum())

Train-Val overlap: 0
Train-Test overlap: 0


## Tokenization

Identical tokenization setup to notebook 02:
- `roberta-base` tokenizer
- `truncation=True, max_length=512`
- Dynamic padding via `DataCollatorWithPadding` (padding deferred to batch collation, not applied per-sequence)
- `label` → `labels` rename required by HuggingFace Trainer

In [6]:
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["combined"],
        truncation=True,
        max_length=512
    )

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

train_ds = Dataset.from_pandas(train_df[["combined", "label"]].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[["combined", "label"]].reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df[["combined", "label"]].reset_index(drop=True))

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")
test_ds  = test_ds.rename_column("label", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/44151 [00:00<?, ? examples/s]

Map:   0%|          | 0/9461 [00:00<?, ? examples/s]

Map:   0%|          | 0/9462 [00:00<?, ? examples/s]

## Load Model

`roberta-base` with 2-class classification head. LOAD REPORT warnings are expected as we are using a binary classifier head instead of MLM head.

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Training Arguments

Identical hyperparameters to notebook 02 — the only variable that changed is the dataset. This is intentional: we want to isolate the effect of semantic deduplication, not confound it with hyperparameter changes.

- `learning_rate=2e-5`, `warmup_steps` = 10% of total steps
- `weight_decay=0.01`
- `fp16=True` — mixed precision
- `load_best_model_at_end=True`, `metric_for_best_model="f1"`

In [8]:
import math


batch_size = 16
epochs = 3
warmup_ratio = 0.1

steps_per_epoch = math.ceil(len(train_ds) / batch_size)
total_steps = steps_per_epoch * epochs
warmup_steps = int(total_steps * warmup_ratio)


args = TrainingArguments(
    output_dir="./roberta-fakenews",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,           
    logging_steps=100,
    report_to="none"
)

## Trainer Setup & Training

**Training curve:**

| Epoch | Train Loss | Val Loss | Val Accuracy | Val F1 |
|-------|------------|----------|--------------|--------|
| 1 | 0.011323 | 0.010590 | 99.78% | 99.78% |
| 2 | 0.004020 | 0.010326 | 99.86% | 99.86% |
| 3 | 0.000012 | 0.007843 | 99.89% | 99.89% |

Val loss decreases across all 3 epochs — the model is not overfitting despite the near-zero training loss at epoch 3. The training loss converging to near-zero on a clean binary classification task with a strong pretrained encoder is expected behaviour, not a red flag.

In [10]:
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.011323,0.010590,0.997780,0.997760
2,0.004020,0.010326,0.998626,0.998614
3,0.000012,0.007843,0.998943,0.998934


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8280, training_loss=0.022385128191559786, metrics={'train_runtime': 2875.3513, 'train_samples_per_second': 46.065, 'train_steps_per_second': 2.88, 'total_flos': 3.484984861559808e+16, 'train_loss': 0.022385128191559786, 'epoch': 3.0})

## Test Set Evaluation

In [14]:
pred = trainer.predict(test_ds)

## Confusion Matrix

```
              Predicted Fake   Predicted Real
Actual Fake        5172                4
Actual Real           7             4279
```

**11 total errors on 9,462 test samples.**

- 4 false positives: real news classified as fake
- 7 false negatives: fake news classified as real

In [15]:
from sklearn.metrics import confusion_matrix 
y_pred = pred.predictions.argmax(axis=-1) 
y_true = pred.label_ids 
print(confusion_matrix(y_true, y_pred))

[[5172    4]
 [   7 4279]]


## Save Model

Saving model and tokenizer to `saved_models/roberta_fake_news` for use in the Gradio demo (notebook 04).

In [17]:
save_path = "saved_models/roberta_fake_news"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('saved_models/roberta_fake_news\\tokenizer_config.json',
 'saved_models/roberta_fake_news\\tokenizer.json')

## Final Results

| Metric | Value |
|--------|-------|
| Test Accuracy | **99.88%** |
| Test Macro F1 | **99.88%** |
| Test Loss | 0.00854 |
| Errors | 11 / 9,462 |

**Comparison with notebook 02 (exact dedup only):** Both runs achieved similar performance, confirming that the ~600 semantic duplicates removed had minimal numerical impact — but the methodology is now cleaner and more defensible.

> **Note:** A mild class imbalance exists (34,502 fake vs 28,572 real in the full dataset), reflected in the slightly different support counts (5,176 fake vs 4,286 real in test). This will be addressed with class weighting in a future.
**Next:** Notebook 04 applies LIME explainability to both the TF-IDF and RoBERTa models to understand *which tokens drive predictions*.